## ANN with MLFLOW

In this project, we will:

* Run a hyperparameter sweep on a traing script
* Compare the results of the runs in Mlflow UI
* Choose the best run and register it as a model
* Deploy the model to a REST API
* Build a container image suitable for deployment to a cloud platform

In [3]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models import infer_signature


In [4]:
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [5]:
# Split the data

train, test = train_test_split(data, test_size=0.25, random_state=42)

In [6]:
train_x = train.drop(['quality'],axis=1).values
train_y= train['quality'].values.ravel()


test_x = test.drop(['quality'],axis=1).values
test_y= test['quality'].values.ravel()

train_x, valid_x, train_y, valid_y = train_test_split(train_x,train_y,test_size=0.2, random_state=42)

signature= infer_signature(train_x, train_y)

In [15]:
# ANN Model

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):

    # define model architecture
    mean= np.mean(train_x, axis=0)
    var= np.var(train_x, axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1)
        ]
    )

    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),epochs=epochs,batch_size=64)

        eval_result = model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rsme = eval_result[1]

        mlflow.log_params(params)
        mlflow.log_metric("eval_rsme",eval_rsme)

        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rsme, "status": STATUS_OK, "model":model}

In [19]:
def objective(params):

    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y
    )

    return result

In [20]:
space={
    "lr":hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum",0.0, 0.1)
}

In [22]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("/wine-quality")
with mlflow.start_run():

    trials= Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )


    best_run =sorted(trials.results, key=lambda x:x["loss"])[0]

    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"],"model",signature=signature)



    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

2026/07/29 20:25:29 INFO mlflow.tracking.fluent: Experiment with name '/wine-quality' does not exist. Creating a new experiment.


Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 8s 194ms/step - loss: 34.5730 - root_mean_squared_error: 5.8799
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 33.2222 - root_mean_squared_error: 5.7639 - val_loss: 33.2210 - val_root_mean_squared_error: 5.7638

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 33.9655 - root_mean_squared_error: 5.8280
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 32.8453 - root_mean_squared_error: 5.7311 - val_loss: 32.8418 - val_root_mean_squared_error: 5.7308

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 33.7736 - root_mean_squared_error: 5.8115
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 32.4728 - root_mean_squared_error: 5.6985 - val_loss: 32.4673 - val_root_mean_squared_error: 5.6980

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 32.6433 - root_mean_squared_error: 5.7134
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 

2026/07/29 20:25:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run abrasive-ox-325 at: http://127.0.0.1:5000/#/experiments/1/runs/e03c91ee8ff7478484e1b796c68bca5c

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 9s 208ms/step - loss: 30.9653 - root_mean_squared_error: 5.5646
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 30.3111 - root_mean_squared_error: 5.5056 - val_loss: 30.1989 - val_root_mean_squared_error: 5.4954

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 28.6038 - root_mean_squared_error: 5.3482
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 29.8291 - root_mean_squared_error: 5.4616 - val_loss: 29.7161 - val_root_mean_squared_error: 5.4513

Epoch 3/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 29.7901 - root_mean_squared_error: 5.4580
46/46 ━━━━━━━━━━━━━━━━━━

2026/07/29 20:25:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

🏃 View run hilarious-roo-923 at: http://127.0.0.1:5000/#/experiments/1/runs/f57f5776cef2449faaa10e445b9d14ea

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                 

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 8s 191ms/step - loss: 33.4610 - root_mean_squared_error: 5.7846
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 30.8164 - root_mean_squared_error: 5.5513 - val_loss: 28.3281 - val_root_mean_squared_error: 5.3224

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 28.5747 - root_mean_squared_error: 5.3455
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 26.3125 - root_mean_squared_error: 5.1296 - val_loss: 24.1774 - val_root_mean_squared_error: 4.9171

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 23.2697 - root_mean_squared_error: 4.8239
46

2026/07/29 20:25:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run auspicious-fox-375 at: http://127.0.0.1:5000/#/experiments/1/runs/db163c1a8dfe4df0a989b4c0707fb490

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

100%|██████████| 4/4 [00:20<00:00,  5.17s/trial, best loss: 1.4388880729675293]

2026/07/29 20:25:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.0019521314979615936), 'momentum': np.float64(0.040798112375640044)}
Best eval rmse: 1.4388880729675293
🏃 View run calm-sheep-126 at: http://127.0.0.1:5000/#/experiments/1/runs/151b555822db487399936caf3a33ee67
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
